In [1]:
!pip install --upgrade pip
!pip install wandb --upgrade
!pip install transformers sentencepiece evaluate jiwer
# !pip install peft -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 23.1.2
    Uninstalling pip-23.1.2:
      Successfully uninstalled pip-23.1.2
  Obtaining dependency information for wandb from https://files.pythonhosted.org/packages/8a/ab/3b6cce52474f273522b4381f4ed93120f1a196d09a8ba65e1f4615fbaa39/wandb-0.15.11-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: wandb
    Found existing installation: wandb 0.15.5
    Uninstalling wandb-0.15.5:
      Successfully uninstalled wandb-0.15.5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 3.0 MB/s eta 0:00:00
  Obtaining dependency information for jiwer from https://files.pythonhosted.org/packages/0d/4f/ee537ab20144811dd99321735ff92ef2b3a3230b77ed7454bed4c44d21fc/jiwer-3.0.3-py3-none-any.whl.metadata


In [2]:
import numpy as np
import pandas as pd
import os
import json
import random
os.makedirs("data", exist_ok=True)

df = pd.read_csv('/kaggle/input/bengaliai-speech/train.csv')
# df['sentence'] = df['sentence'].apply(remove_special_characters,)
train_df = df[df['split'] == 'train'][['id', 'sentence']].sample(frac = 1, ).reset_index(drop=True).iloc[:100000].reset_index(drop=True)
val_df   = df[df['split'] == 'valid'][['id', 'sentence']].sample(frac = 1).reset_index(drop=True).iloc[:2000].reset_index(drop=True)
# new_train = {'id': [], 'sentence': []}
# train_valu = train_df.values
# for i in range(0, len(train_valu), 2):
#     r = random.random()
#     if r < 0.5:
#         new_id = f'{train_valu[i][0]}|||{train_valu[i+1][0]}'
#         new_sent = train_valu[i][1] + ' ' + train_valu[i+1][1]
#         new_train['id'].append(new_id)
#         new_train['sentence'].append(new_sent)
#     else:
#         for j in range(2):
#             new_train['id'].append(train_valu[i+j][0])
#             new_train['sentence'].append(train_valu[i+j][1])
    
# new_train = pd.DataFrame().from_records(new_train)

# new_val = {'id': [], 'sentence': []}
# val_valu = val_df.values
# for i in range(0, len(val_valu), 2):
#     new_id = f'{val_valu[i][0]}|||{val_valu[i+1][0]}'
#     new_sent = val_valu[i][1] + ' ' + val_valu[i+1][1]
#     new_val['id'].append(new_id)
#     new_val['sentence'].append(new_sent)
    
# new_val = pd.DataFrame().from_records(new_val)

# NUM_OF_ROWS = new_train.shape[0]
# new_train.to_csv('data/train.csv', index=False)
# new_val.to_csv('data/valid.csv', index=False)

train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/valid.csv', index=False)
NUM_OF_ROWS = train_df.shape[0]



del df, train_df, val_df #, train_valu, new_train, new_val, val_valu

In [3]:
!ls

data


In [4]:
import os
import json
from transformers import SpeechT5Tokenizer, SpeechT5FeatureExtractor, SpeechT5Processor

model_id = "microsoft/speecht5_asr"

feature_extractor = SpeechT5FeatureExtractor.from_pretrained(model_id)

vocab_path = '/kaggle/input/helper-for-bangla/tokenizer/spm.model'

tokenizer = SpeechT5Tokenizer(vocab_path, unk_token="<unk>", pad_token="<pad>", eos_token='</s>', bos_token='<s>', )

processor = SpeechT5Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

processor.save_pretrained('speecht5_asr')

del feature_extractor, processor, SpeechT5Tokenizer, SpeechT5FeatureExtractor, SpeechT5Processor

In [20]:
import librosa
import numpy as np
def numarize(batch, valid=0):
    sr = 16000
    def load_mp3(file_path):
        file_paths = file_path.split('|||')
        audios = []
        for f in file_paths:
            au, _ = librosa.load(f'/kaggle/input/bengaliai-speech/train_mp3s/{f}.mp3', sr=sr)
            audios.append(au)
            audios.append(np.zeros((500,)))
        audios = np.hstack(audios)
        return audios
    
    audio = list(map(load_mp3, batch['id']))
    # batched output is "un-batched"
    inputs = {'input_values': audio, 'valid': [valid]*len(audio)}
    if valid:
        inputs["labels"] = batch["sentence"]
    else:
        inputs["labels"] = processor(text=batch["sentence"]).input_ids

    return inputs

class DataCollatorWav2VecWithPadding:
    def __init__(self, processor, model=None, padding = True, max_length = None, max_length_labels = None,
                 pad_to_multiple_of = None, pad_to_multiple_of_labels = None, truncation=None, return_tensors='pt'):
        self.processor = processor
        self.model = model
        self.padding = padding
        self.max_length = max_length
        self.max_length_labels = max_length_labels
        self.pad_to_multiple_of = pad_to_multiple_of
        self.pad_to_multiple_of_labels = pad_to_multiple_of_labels
        self.truncation = truncation
        self.return_tensors = return_tensors
    
    def __call__(self, features, return_tensors=None):
        if return_tensors is not None:
            self.return_tensors = return_tensors

        label_name = "label" if "label" in features[0].keys() else "labels"
        
        no_labels_features = [v for feature in features for k, v in feature.items() if k != label_name and k != 'valid']
        valid = any([v for feature in features for k, v in feature.items() if k == 'valid'])
        sr = 16000
        if valid:
            labels = [v  for feature in features for k, v in feature.items() if k == label_name] if label_name in features[0].keys() else None
            batch = self.processor.feature_extractor(no_labels_features,
                                                     sampling_rate=sr,
                                                     padding='max_length', 
                                                     max_length=int(5*16000),
                                                     truncation=True,
                                                     return_tensors=self.return_tensors)
            if labels is not None:
                labels_batch = self.processor(text=labels, 
                                              padding='max_length',
                                              max_length=120,
                                              truncation=True,
                                              pad_to_multiple_of=self.pad_to_multiple_of_labels,
                                              return_tensors=self.return_tensors) 
                
        
        else:
            labels = [{'input_ids': v for k, v in feature.items() if k == label_name} for feature in features ] if label_name in features[0].keys() else None
            batch = self.processor.feature_extractor(no_labels_features,
                                                     sampling_rate=sr,
                                                       padding=self.padding, 
                                                       max_length=self.max_length,
                                                       truncation=self.truncation,
                                                       pad_to_multiple_of=self.pad_to_multiple_of,
                                                       return_tensors=self.return_tensors)
            if labels is not None:
                labels_batch = self.processor.tokenizer.pad(labels, 
                                             padding=self.padding,
                                             max_length=self.max_length_labels,
    #                                          truncation=self.truncation,
                                             pad_to_multiple_of=self.pad_to_multiple_of_labels,
                                             return_tensors=self.return_tensors)   

        
        # replace padding with -100 to ignore loss correctly

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels   
        
        return batch
        

In [6]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}"
    )

In [43]:
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText
from datasets import load_dataset

# model_id = "microsoft/speecht5_asr"
model_id = 'speecht5_asr'
processor = SpeechT5Processor.from_pretrained(model_id)

model_id = "microsoft/speecht5_asr"
model = SpeechT5ForSpeechToText.from_pretrained(model_id, vocab_size=processor.tokenizer.vocab_size, ignore_mismatched_sizes=True, use_guided_attention_loss=False)
model.speecht5.decoder.prenet.embed_tokens.padding_idx = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.use_cache = False

print_trainable_parameters(model)

Some weights of SpeechT5ForSpeechToText were not initialized from the model checkpoint at microsoft/speecht5_asr and are newly initialized because the shapes did not match:
- speecht5.decoder.prenet.embed_tokens.weight: found shape torch.Size([81, 768]) in the checkpoint and torch.Size([86, 768]) in the model instantiated
- text_decoder_postnet.lm_head.weight: found shape torch.Size([81, 768]) in the checkpoint and torch.Size([86, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 151168896 || all params: 154592640 || trainable%: 97.79


In [15]:
model.speecht5.encoder.requires_grad_(False)
model.speecht5.decoder.requires_grad_(True);

In [16]:
# model.speecht5.encoder.requires_grad_(False)
# for n, p in model.speecht5.decoder.wrapped_decoder.named_parameters():
#     if '3' in n or '4' in n or '5' in n:
#         p.requires_grad_(True)
#     else:
#         p.requires_grad_(False)
        
# for n, p in model.speecht5.encoder.named_parameters():
#     if 'pos' in n or '8' in n or '9' in n or '10' in n or '11' in n:
#         p.requires_grad_(True)
#     else:
#         p.requires_grad_(False)

In [17]:
print_trainable_parameters(model)

trainable params: 57125376 || all params: 154592640 || trainable%: 36.95


In [18]:
import os
# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="bngali_ASR"

# save your trained model checkpoint to wandb
os.environ["WANDB_LOG_MODEL"]="true"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="False"

os.environ["WANDB_API_KEY"] = 'b8ad1673354b49d42bf0bfd9436537326dd350cc'

# os.environ["WANDB_LOG_MODEL"] = 'true'

# os.environ["WANDB_WATCH"] = 'all'

In [19]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset
from evaluate import load
import numpy as np
import torch
import wandb
from functools import partial

In [21]:
batch_size = 20

train_data = load_dataset('data', split='train', streaming=False)
# train_colleter = DataCollatorWav2txtStreaming(processor, model=model, padding='longest', return_tensors='pt')

maped_train = train_data.map(numarize, remove_columns=['id', 'sentence'], batched=True, batch_size=batch_size, num_proc=2)
# maped_train = maped_train.with_format("torch")

val_data = load_dataset('data', split='validation', streaming=False)
# val_colleter = DataCollatorWav2txtWithPadding(processor, model=model, padding='longest', return_tensors='pt')
maped_val = val_data.map(partial(numarize, valid=1), remove_columns=['id', 'sentence'], batched=True, batch_size=batch_size, num_proc=2)
# maped_val = maped_val.with_format("torch")

val_colleter = DataCollatorWav2VecWithPadding(processor, model=model, padding='longest', return_tensors='pt')

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/datasets/packaged_modules/csv/csv.py:154: FutureWarning: the 'mangle_dupe_cols' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'mangle_dupe_cols'
  csv_file_reader = pd.read_csv(file, iterator=True, dtype=dtype, **self.config.read_csv_kwargs)


Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/data-65b597b894438c3f/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519. Subsequent calls will reuse this data.
  

/opt/conda/lib/python3.10/site-packages/datasets/packaged_modules/csv/csv.py:154: FutureWarning: the 'mangle_dupe_cols' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'mangle_dupe_cols'
  csv_file_reader = pd.read_csv(file, iterator=True, dtype=dtype, **self.config.read_csv_kwargs)


#0:   0%|          | 0/2500 [00:00<?, ?ba/s]

#1:   0%|          | 0/2500 [00:00<?, ?ba/s]

#0:   0%|          | 0/50 [00:00<?, ?ba/s]

#1:   0%|          | 0/50 [00:00<?, ?ba/s]

In [22]:
def train(data_size=None, batch_size=1, acc_steps=1, eval_batch_size=1, eval_acc_steps=1, epochs=1):
    
    wer = load("wer")
    def wer_metric(pred):
        pred_ids = pred.predictions
        pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        # we do not want to group tokens when computing the metrics
        label_str = processor.batch_decode(pred.label_ids, skip_special_tokens=True)

        wer_metric = wer.compute(predictions=pred_str, references=label_str)

        return {"wer": wer_metric}
    def preprocess_logits_for_metrics(logits, labels):
        """
        Original Trainer may have a memory leak. 
        This is a workaround to avoid storing too many tensors that are not needed.
        """
        pred_ids = torch.argmax(logits[0], dim=-1)
        return pred_ids
    args = Seq2SeqTrainingArguments(
                                output_dir='model_checkpoint',
                                do_train=True,
                                do_eval=True,
                                per_device_train_batch_size=batch_size,
                                per_device_eval_batch_size=eval_batch_size,
                                gradient_accumulation_steps=acc_steps,
                                eval_accumulation_steps=eval_acc_steps,
                                dataloader_num_workers=2,
                                learning_rate=5e-5,

#                                 max_steps = data_size // (batch_size*acc_steps),
#                                 adafactor=True,
                                weight_decay=0.009,
                                max_grad_norm=0.5,
#                                 label_smoothing_factor=0.1,
                                num_train_epochs=epochs,
                                warmup_steps=300,
                                dataloader_drop_last=True,
                                save_steps=500,
                                eval_steps=250,
                                logging_steps=25,
#                                     eval_delay=500,
                                evaluation_strategy='steps',
                                logging_strategy='steps',
                                lr_scheduler_type='cosine_with_restarts',
                                logging_first_step=True,
                                fp16=True,
                                fp16_full_eval=True,
                                gradient_checkpointing=True,
                                save_total_limit=4,
                                load_best_model_at_end=True,
                                metric_for_best_model="wer",
                                resume_from_checkpoint=False,
#                                 log_level='debug',
                                sortish_sampler=True,
#                                 predict_with_generate=True,
#                                 generation_max_length=80,
                                remove_unused_columns=False,
#                                 label_names=['labels'],
                                run_name='t5-full-decoder',
                                report_to='wandb'
    )
    
    
    doing = Seq2SeqTrainer(model, args, data_collator=val_colleter, 
                           train_dataset=maped_train,
                           eval_dataset=maped_val,
                           compute_metrics=wer_metric,
                           tokenizer=processor.feature_extractor,
                           preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                          )
    
    train_result = doing.train(resume_from_checkpoint=False)
    wandb.finish()
    return doing, train_result

# ['linear', 'cosine', 'cosine_with_restarts', 'polynomial', 'constant', 'constant_with_warmup', 'inverse_sqrt', 'reduce_lr_on_plateau']

In [23]:
trainer, train_result = train(data_size=None, batch_size=8, acc_steps=4, eval_batch_size=10, eval_acc_steps=15, epochs=2)

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
wandb: Currently logged in as: sebaeymohamed43. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Wer
250,4.004900,2.873085,1.139101
500,3.198800,2.429987,1.098589
750,2.392400,1.705319,0.977713
1000,2.055000,1.409319,1.001921
1250,1.762300,1.300030,0.941538
1500,1.688000,1.231504,0.895153
1750,1.661800,1.172832,0.876983
2000,1.494900,1.098577,0.912115
2250,1.430000,1.052817,0.876434
2500,1.482300,1.001870,0.887687


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 trainer, train_result = train(data_size=None, batch_size=8, acc_steps=4, eval_batch_size     │
│   2                                                                                              │
│                                                                                                  │
│ in train:75                                                                                      │
│                                                                                                  │
│   72 │   │   │   │   │   │      preprocess_logits_for_metrics=preprocess_logits_for_metrics,     │
│   73 │   │   │   │   │   │     )                                                                 │
│   74 │                                                                                           │
│ ❱ 75 │   train_result = doing.train(resume_from_checkpoint=False)                                │
│   76 │   wandb.finish()                                                                          │
│   77 │   return doing, train_result                                                              │
│   78                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/trainer.py:1645 in train                    │
│                                                                                                  │
│   1642 │   │   inner_training_loop = find_executable_batch_size(                                 │
│   1643 │   │   │   self._inner_training_loop, self._train_batch_size, args.auto_find_batch_size  │
│   1644 │   │   )                                                                                 │
│ ❱ 1645 │   │   return inner_training_loop(                                                       │
│   1646 │   │   │   args=args,                                                                    │
│   1647 │   │   │   resume_from_checkpoint=resume_from_checkpoint,                                │
│   1648 │   │   │   trial=trial,                                                                  │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/trainer.py:1938 in _inner_training_loop     │
│                                                                                                  │
│   1935 │   │   │   │   │   self.control = self.callback_handler.on_step_begin(args, self.state,  │
│   1936 │   │   │   │                                                                             │
│   1937 │   │   │   │   with self.accelerator.accumulate(model):                                  │
│ ❱ 1938 │   │   │   │   │   tr_loss_step = self.training_step(model, inputs)                      │
│   1939 │   │   │   │                                                                             │
│   1940 │   │   │   │   if (                                                                      │
│   1941 │   │   │   │   │   args.logging_nan_inf_filter                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/trainer.py:2751 in training_step            │
│                                                                                                  │
│   2748 │   │   Return:                                                                           │
│   2749 │   │   │   `torch.Tensor`: The tensor with training

In [ ]:
trainer.save_model('best_model')
processor.save_pretrained('best_model')

https://discuss.huggingface.co/t/cuda-out-of-memory-when-using-trainer-with-compute-metrics/2941

In [ ]:
5

In [29]:
!ls model_checkpoint/checkpoint-4000

config.json		preprocessor_config.json  scheduler.pt
generation_config.json	pytorch_model.bin	  trainer_state.json
optimizer.pt		rng_state.pth		  training_args.bin


In [33]:
!ls speecht5_asr

config.json		  pytorch_model.bin	   tokenizer_config.json
generation_config.json	  special_tokens_map.json
preprocessor_config.json  spm_char.model


In [32]:
!cp model_checkpoint/checkpoint-4000/config.json model_checkpoint/checkpoint-4000/pytorch_model.bin model_checkpoint/checkpoint-4000/generation_config.json speecht5_asr/

In [30]:
model.save_pretrained('twst')

In [34]:
!ls

data  model_checkpoint	speecht5_asr  twst  wandb


In [35]:
!rm -r model_checkpoint/ twst/ wandb/

# at inference

In [75]:
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText
from datasets import load_dataset

model_id = 'speecht5_asr'
inf_processor = SpeechT5Processor.from_pretrained(model_id)

inf_model = SpeechT5ForSpeechToText.from_pretrained(model_id, vocab_size=inf_processor.tokenizer.vocab_size)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│   4 model_id = 'speecht5_asr'                                                                    │
│   5 inf_processor = SpeechT5Processor.from_pretrained(model_id)                                  │
│   6                                                                                              │
│ ❱ 7 inf_model = SpeechT5ForSpeechToText.from_pretrained(model_id, vocab_size=inf_processor.t     │
│   8                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/modeling_utils.py:2881 in from_pretrained   │
│                                                                                                  │
│   2878 │   │   │   │   mismatched_keys,                                                          │
│   2879 │   │   │   │   offload_index,                                                            │
│   2880 │   │   │   │   error_msgs,                                                               │
│ ❱ 2881 │   │   │   ) = cls._load_pretrained_model(                                               │
│   2882 │   │   │   │   model,                                                                    │
│   2883 │   │   │   │   state_dict,                                                               │
│   2884 │   │   │   │   loaded_state_dict_keys,  # XXX: rename?                                   │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/modeling_utils.py:3278 in                   │
│ _load_pretrained_model                                                                           │
│                                                                                                  │
│   3275 │   │   │   │   error_msg += (                                                            │
│   3276 │   │   │   │   │   "\n\tYou may consider adding `ignore_mismatched_sizes=True` in the m  │
│   3277 │   │   │   │   )                                                                         │
│ ❱ 3278 │   │   │   raise RuntimeError(f"Error(s) in loading state_dict for {model.__class__.__n  │
│   3279 │   │                                                                                     │
│   3280 │   │   if is_quantized:                                                                  │
│   3281 │   │   │   unexpected_keys = [elem for elem in unexpected_keys if "SCB" not in elem]     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
RuntimeError: Error(s) in loading state_dict for SpeechT5ForSpeechToText:
        size mismatch for speecht5.decoder.prenet.embed_positions.weights: copying a param with shape 
torch.Size([454, 768]) from checkpoint, the shape in current model is torch.Size([453, 768]).
        You may consider adding `ignore_mismatched_sizes=True` in the model `from_pretrained` method.

In [52]:
model.speecht5.decoder.prenet.embed_positions.weights.shape

torch.Size([454, 768])

In [71]:
x = torch.load('speecht5_asr/pytorch_model.bin')
for i in x.keys():
    if 'decoder' in i and 'embed_positions' in i:
        print(x[i].size())
        k = i

torch.Size([454, 768])


In [68]:
xx = model.state_dict()
for i in xx.keys():
    if 'decoder' in i and 'embed_positions' in i:
        print(xx[i].size())

torch.Size([454, 768])


In [74]:
model.speecht5.decoder.prenet.embed_positions.weights.data = x[k].data

In [78]:
SpeechT5ForSpeechToText()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 SpeechT5ForSpeechToText()                                                                    │
│   2                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: SpeechT5ForSpeechToText.__init__() missing 1 required positional argument: 'config'